# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# View the metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set, field, and column is referenced by its `@id` in the Croissant schema for consistency.

In [ ]:
# List all record sets available in the dataset, using the @id
print("Available Record Sets (by @id):")
record_sets = dataset.metadata.recordSet
if record_sets is None or len(record_sets) == 0:
    print("No record sets defined in the Croissant schema metadata. Attempting to infer record set from data...")
    # Attempt to infer the primary record set id from records() generator
    # mlcroissant often autogenerates the main table @id as "table" or similar if only single table
    # Let's try loading the records and inspect the keys
    records_preview = list(dataset.records(limit=1))
    if records_preview:
        available_fields = list(records_preview[0].keys())
        print(f"Single tabular record set detected with fields: {available_fields}")
        # We'll call the inferred record set id as 'table' (as common in mlcroissant)
        record_sets = ["table"]
    else:
        print("No records found.")
        record_sets = []
else:
    for rs in record_sets:
        print(f" - {rs['@id']}: {rs.get('name', '(no name)')}")

# For each record set, list its fields by @id
for rec_id in record_sets:
    if isinstance(rec_id, dict):
        # structured, as in Croissant schema
        rec_set_id = rec_id['@id']
    else:
        # fallback: just id string
        rec_set_id = rec_id
    print(f"\nFields for record set '@id' = {rec_set_id}:")
    try:
        recset_obj = dataset.metadata.record_set(rec_set_id)
        if recset_obj is not None and hasattr(recset_obj, 'field'):
            for f in recset_obj.field:
                print(f" - Field @id: {f['@id']}, with dataType: {f.get('dataType', '(unknown)')}")
        else:
            # May be a generic file-based tabular record set
            sample_records = list(dataset.records(record_set=rec_set_id, limit=1))
            if sample_records:
                print("Fields (inferred from data):")
                print(" - " + '\n - '.join(sample_records[0].keys()))
            else:
                print("(No fields found in data sample.)")
    except Exception as e:
        print(f"Could not fetch fields for {rec_set_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

We demonstrate this by loading the first (or only) detected record set using its `@id`, and show the column (field) names from the loaded DataFrame.

In [ ]:
# Extract data from each record set (by @id)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

dataframes = {}

# If there are no declared recordSet in metadata, use the inferred 'table' id
record_set_ids = []
if isinstance(record_sets, list) and len(record_sets) > 0:
    # Croissant record sets may be dicts with @id or just id strings
    for rs in record_sets:
        if isinstance(rs, dict):
            rec_id = rs['@id']
        else:
            rec_id = rs
        record_set_ids.append(rec_id)
else:
    record_set_ids = ['table']  # fallback for single table datasets

# Load all record sets into dataframes
for rec_id in record_set_ids:
    print(f"Loading records for record set: '{rec_id}'")
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records with fields: {df.columns.tolist()}")

# Preview the columns of the first record set
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id:
    print(f"\nColumns in DataFrame for record set '@id' = {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Here, we select a numeric field by its column/@id for analysis, filter based on a threshold, and normalize the field.

In [ ]:
# Select a numeric field for analysis by its column/@id
df = dataframes[primary_record_set_id]
print("Available columns:", df.columns.tolist())

# Attempt to select a likely numeric field
import numpy as np
numeric_candidates = [col for col in df.columns if df[col].dtype in ('int64', 'float64') or np.issubdtype(df[col].dtype, np.number)]
if not numeric_candidates:
    # try to find columns with numeric-like content
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    numeric_candidates = [col for col in df.columns if df[col].dtype in ('int64', 'float64') or np.issubdtype(df[col].dtype, np.number)]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # Default to first column if no numeric found
    numeric_field_id = df.columns[0]

print(f"Selected numeric field for filtering: {numeric_field_id}")
//--- Use a threshold suitable for the data ---//

if np.issubdtype(df[numeric_field_id].dtype, np.number):
    threshold = np.percentile(df[numeric_field_id].dropna(), 75)  # use 75th percentile as sample threshold
else:
    print(f"Column {numeric_field_id} is not recognized as numeric.")
    threshold = None

if threshold is not None:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Attempt grouping by a categorical field (@id)
    group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    group_field_id = group_candidates[0] if group_candidates else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id} (top 5):")
        print(grouped_df.head())
else:
    print("No numeric field suitable for filtering in this dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. 

We'll show a histogram for the selected numeric field, and (if applicable) a boxplot by some category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id and numeric_field_id in df:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Optional: boxplot by group_field_id if it exists
    if 'group_field_id' in locals() and group_field_id and group_field_id in df:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR\(^2\) dataset '[Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)' via its Croissant schema using the `mlcroissant` library.

- We explored the dataset's metadata, available record sets, and fields, referencing all entities by their `@id`s for clarity and reproducibility.
- We extracted tabular records into Pandas DataFrames, performed basic exploratory analyses including filtering, normalization, and aggregation by categorical attribute.
- Visualizations illustrated the distributions and possible relationships within the data.

This process demonstrates a standardized, reproducible approach to FAIR dataset exploration using Croissant schemas and the `mlcroissant` Python library.